<a href="https://colab.research.google.com/github/FabsRobles/Practica2-IA/blob/main/notebook/Titanic_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 1. Comprensión del dominio del dataset.

En esta primera parte se hizo un análisis de los datos que aparecen en nuestro dataset, se identificaron tipos de datos, valores y celdas vacías.

In [2]:
import pandas as pd

url="https://raw.githubusercontent.com/FabsRobles/Practica2-IA/refs/heads/main/train.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Teniendo la tabla de valores ahora inspeccionamos los tipos de datos con los que contamos, qué contiene cáda columna y sus valores importantes.

### 1. ¿Cuál es el nombre de las columnas?

In [3]:
df.columns.tolist()

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

### 2. ¿Qué representa cada columna?

In [8]:
datos={
    'Columna': ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
    'Descripción': ['Identificador único del pasajero', 'Indica si el pasajero sobrevivió (1) o no (0)', 'Clase del boleto: Primera, Segunda o Tercera0', 'Nombre del pasajero', 'Sexo(male/female)', 'Edad en años', 'Cuántos hermanos o espos@ iban abordo', 'Cuántos padres o hijos iban abordo', 'Número de boleto', 'Lo que pagó por el boleto', 'Número de camarote', 'Puerto donde abordó: C,Q o S'],
    'Tipo de dato': ['Entero', 'Entero', 'Entero', 'Texto', 'Texto', 'Flotante', 'Entero', 'Entero', 'Texto', 'Flotante', 'Texto', 'Texto']
}

df=pd.DataFrame(datos)

df

,Columna,Descripción,Tipo de dato
0,PassengerId,Identificador único del pasajero,Entero
1,Survived,Indica si el pasajero sobrevivió (1) o no (0),Entero
2,Pclass,"Clase del boleto: Primera, Segunda o Tercera0",Entero
3,Name,Nombre del pasajero,Texto
4,Sex,Sexo(male/female),Texto
5,Age,Edad en años,Flotante
6,SibSp,Cuántos hermanos o espos@ iban abordo,Entero
7,Parch,Cuántos padres o hijos iban abordo,Entero
8,Ticket,Número de boleto,Texto
9,Fare,Lo que pagó por el boleto,Flotante


### 3. ¿Cuántos datos faltantes hay en cada columna?

In [5]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


### 4. ¿Hay filas repetidas?

In [7]:
df.duplicated().sum()

np.int64(0)

Después de inspeccionar en qué condiciones se encuentra nuestro dataset, respondemos a las siguientes preguntas.

*   ¿Qué representa cada fila?
> Cada fila de nuestro dataset representa a un pasajero del Titanic.

*   ¿La variable objetivo estaría disponible en un escenario real?
> No, ya que la unica forma de saber si alguien sobrevivió es después del hundimiento, no antes.


*   ¿Existen identificadores o variables con fuga de información?
> La variable Survived contiene la información que requerimos para el análisis, sin embargo en un caso real, esa variable solo estará llena después del hundimiento del barco.


*   ¿Qué decisiones podrían afectar de forma desigual a distintos grupos?
> Si un modelo usara las variables Sex, Age, Pclass, podría favorecer o perjudicar a mujeres, niños o personas de clase baja.







# Actividad 2. Data Profiling

Antes de comenzar dividimos las columnas en grupos de la siguiente manera.

In [9]:
col_num=["Age", "SibSp", "Parch", "Fare"]
col_cat=["Survived", "Pclass", "Sex", "Embarked", "Cabin"]
col_id= ["PassengerId", "Name", "Ticket"]

Lo dividimos de esta forma ya que a pesar de que las variables como "Survived" y "Pclass" parecen números son categorías, ya que no las podemos promediar, mientras que "PassengerId", "Name", "Ticket" no se repiten lo suficiente para considerarlas una categoría.

### Parte 1. Tipo de dato, valores válidos y ausentes

In [20]:
def validos_ausentes(columnas):
  return pd.DataFrame({
      "tipo": df[columnas].dtypes.astype(str),
      "válidos": df[columnas].notna().sum(), #cuantos pasajeros si tienen dato
      "ausentes": df[columnas].isna().sum() #pasajeros que no tienen celda vacía
  })

print("Numericas")
display(validos_ausentes(col_num))

print("Categóricas")
display(validos_ausentes(col_cat))

Numericas


,tipo,válidos,ausentes
Age,float64,714,177
SibSp,int64,891,0
Parch,int64,891,0
Fare,float64,891,0


Categóricas


,tipo,válidos,ausentes
Survived,int64,891,0
Pclass,int64,891,0
Sex,object,891,0
Embarked,object,889,2
Cabin,object,204,687


### Parte 2. Porcentaje de ausencia y valores únicos

In [14]:
# Volvemos a cargar el dataset original para restaurar la variable 'df'
df = pd.read_csv("https://raw.githubusercontent.com/FabsRobles/Practica2-IA/refs/heads/main/train.csv")

def ausencia_unicos(columnas):
  return pd.DataFrame({
      "% ausentes": (df[columnas].isna().mean()*100).round(2),
      "valores únicos": df[columnas].nunique()
  })

print("Numericas")
display(ausencia_unicos(col_num))

print("Categoricas")
display(ausencia_unicos(col_cat))

Numericas


,% ausentes,valores únicos
Age,19.87,88
SibSp,0.00,7
Parch,0.00,7
Fare,0.00,248


Categoricas


,% ausentes,valores únicos
Survived,0.00,2
Pclass,0.00,3
Sex,0.00,2
Embarked,0.22,3
Cabin,77.10,147


Para esta primera parte buscamos las columnas que tienen datos vacíos y cuántos, con df.isna().sum(), df.isna().mean()*100. De esta manera encontramos que "Cabin" (77.10%), "Age" (19.87%) y "Embarked" (0.22%), tienen celdas vacías, mientras que las demás columnas cuentan con datos completos.

### Parte 3. Mínimo, máximo, media, mediana y desviación.

In [21]:
resumen_p3 = df[col_num].agg(["min", "max", "mean", "median", "std"]).T.round(2)
resumen_p3.columns = ["mínimo", "máximo", "media", "mediana", "desv. estándar"]
resumen_p3

,mínimo,máximo,media,mediana,desv. estándar
Age,0.42,80.00,29.70,28.00,14.53
SibSp,0.00,8.00,0.52,0.00,1.10
Parch,0.00,6.00,0.38,0.00,0.81
Fare,0.00,512.33,32.20,14.45,49.69


La tabla anterior se hizo unicamente con los valores númericos con los que contaba nuestro dataset, es decir, los que podemos promediar.

Podemos observar que en la variable "Age" la media y la mediana no difieren mucho, por lo que no hay un sesgo grande, sin embargo en "Fare" la media duplica a la mediana y el valor máximo es muy grande, indicando que hay valores extremos, las varibles "SibSp" y "Parch" indican que la mayoría de los pasajeros viajaba sin familiares como padres, hermanos, hijos o esposos.

### Parte 4. Percentiles y asimetría

In [23]:
percentiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]

p4 = df[col_num].quantile(percentiles).T
p4.columns = ["1%", "5%", "25%", "50%", "75%", "95%", "99%"]
p4["asimetría"] = df[col_num].skew()
p4.round(2)

,1%,5%,25%,50%,75%,95%,99%,asimetría
Age,1.0,4.00,20.12,28.00,38.0,56.00,65.87,0.39
SibSp,0.0,0.00,0.00,0.00,1.0,3.00,5.00,3.70
Parch,0.0,0.00,0.00,0.00,0.0,2.00,4.00,2.75
Fare,0.0,7.22,7.91,14.45,31.0,112.08,249.01,4.79


Del desarrollo de esta parte podemos observar mediante los valores de asimetría que "Age" está bastante equilibrada (asimetría=0.39). La mitad de los pasajeros tienen entre 20 y 38 años. Por otro lado "Fare" es la verdadera protagonista, ya que cuenta no una asímetría de 4.79, lo cual nos indica que está muy estirada hacia la derecha, el 99% pagó 249, sin embargo el valor máximo que hallamos fue de 512, hay una diferencia enorme, lo que nos indica que casi todos pagaron poco, y un pequeño grupo pagó muchisimo.

## Hallazgos importantes.

*   "Cabin" está vacía en un 77.10% y tiene 147 valores distintos, por lo que tratar de rellenarla o como categoría se vuelva bastante inservible
*   "Embarked" solo tiene 2 elementos vacíos, si los rellenamos no representarían ningún peligro para nuestro análisis.
*   "Fare" tiene una media de 32.20 contra una mediana de 14.45 y un máximo de 512.33, lo que indica posibles valores extremos.
*   "Fare" además, tiene un mínimo de 0 el cuál hay que tomar en cuenta y revisar.
*   "Age" tiene una asimetria aproximada de 0.39, lo que indica que los valores son bastante equilibrados.
*   "Fare", por otro lado, tiene una asimetría de 4.79, pudimos observar que su media es de 32, pero solo el 75% de los pasajeros pagó por este precio, sin embargo hubieron algunos que pagaron 512, un valor muy muy alto.







